# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout (`holdout.csv`), 5-fold stratified CV on training rows, model chosen by mean CV roc_auc, holdout scored once.

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='083821b17b5c7fdf81de46633ca08a7d601bd8266b269faa3911938d26bf66bf'
df=pd.read_csv('data.csv'); target='claim_within_year'
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
is_hold=np.zeros(len(df),bool); is_hold[hold_idx]=True
print(len(df), is_hold.sum(), (~is_hold).sum())

## Features
`device_serial` is excluded (identifier, per protocol). `channel`, `device_age_years`, `price_eur` are known at sale; `daily_hours` is reported at registration, i.e. before the outcome window, so all four are usable at prediction time.

In [ ]:
cat=['channel']; num=['device_age_years','price_eur','daily_hours']
features=cat+num
X=df[features]; y=df[target]
Xtr,ytr=X[~is_hold],y[~is_hold]; Xho,yho=X[is_hold],y[is_hold]
print(ytr.mean(), yho.mean())

## Candidates (all named before any scoring)
Preprocessing lives inside each pipeline, so it is fit within CV folds.

In [ ]:
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),('n',StandardScaler() if scale else 'passthrough',num)])
candidates={
 'baseline_prior': make_pipeline(prep(False),DummyClassifier(strategy='prior')),
 'logreg_C1': make_pipeline(prep(True),LogisticRegression(C=1.0,max_iter=1000)),
 'logreg_C0.1': make_pipeline(prep(True),LogisticRegression(C=0.1,max_iter=1000)),
 'rf_300_leaf20': make_pipeline(prep(False),RandomForestClassifier(300,min_samples_leaf=20,random_state=0,n_jobs=-1)),
 'hgb_depth3': make_pipeline(prep(False),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)),
}
print(list(candidates))

## Cross-validation on training rows only

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=0)
cv_scores={n:cross_val_score(m,Xtr,ytr,cv=cv,scoring='roc_auc').tolist() for n,m in candidates.items()}
for n,s in cv_scores.items(): print(f'{n:16s} mean={np.mean(s):.4f} sd={np.std(s):.4f}')
chosen=max(cv_scores,key=lambda n:np.mean(cv_scores[n])); print('chosen:',chosen)

## Refit on all training rows, score once on holdout

In [ ]:
model=candidates[chosen].fit(Xtr,ytr)
holdout_auc=roc_auc_score(yho,model.predict_proba(Xho)[:,1]); print('holdout roc_auc',holdout_auc)

In [ ]:
train_rows=np.flatnonzero(~is_hold)
folds=[[train_rows[a].tolist(),train_rows[b].tolist()] for a,b in cv.split(Xtr,ytr)]
evidence={'features':features,'candidates':list(candidates),'cv_folds':folds,
 'cv_scores':cv_scores,'chosen':chosen,'holdout_rows':np.flatnonzero(is_hold).tolist(),
 'holdout_scores':{chosen:holdout_auc}}
json.dump(evidence,open('evidence.json','w'))